# 03 — Periodic Phase Encoding

노트북은 Step 3 실험 실행만 담당하고, 공통 학습/평가 로직은 `pcdp/`로 분리했습니다.


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config

In [ ]:
import torch

from pcdp.configs import get_experiment_config
from pcdp.experiment_plots import plot_action_chunks, plot_loss_curve
from pcdp.experiment_runner import (
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    sample_frequency_variants,
    train_or_load_checkpoint,
)
from pcdp.phase import periodic_offline_frequencies
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('periodic_phase')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 3. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 4. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 5. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 6. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)


## 7. Offline Phase Sensitivity

In [ ]:
freqs, labels = periodic_offline_frequencies(data)
ep_obs = val_ds[0]['obs'].unsqueeze(0)
samples_by_freq = sample_frequency_variants(
    model, ema, ns_config, ep_obs, data, sample_cond_fn, freqs, labels,
    device=device, num_inference_steps=NUM_INFERENCE_STEPS, dt=cfg.evaluation.dt, seed=data['seed'],
)
plot_action_chunks(
    samples_by_freq, FIGURES_DIR / 'phase_periodic_sensitivity.png',
    title='Periodic DP — same obs, different rollout phase frequencies', act_dim=data['ACT_DIM'],
)
